# Working with Data

The [Spatial DataFrame](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#spatialdataframe) (SDF) creates a simple, intutive object that can easily manipulate geometric and attribute data without forcing you to use a full-fledged copy of any source information. The SDF is based on data structures naturally suited to data analysis, with natural operations for filtering and inpsecting subsets of values that are fundamental to statistical manipulations. 

Data can be read from many **sources**, including shapefiles, [Pandas](https://pandas.pydata.org/) [DataFrames](http://pandas.pydata.org/pandas-docs/stable/dsintro.html#dataframe), feature classes, GeoJSON, and Feature Layers.

This document outlines some fundamentals of using the Spatial DataFrame object for working with GIS data.

* [Accessing Published Data](#Accessing-Published-Data)
 * [Reading Service Data](#Reading-Service-Data)
 * [Reading Feature Layer Data](#Reading-Feature-Layer-Data)
   * [Examining Feature Layer Content](#Example:-Examining-Feature-Layer-content)
   * [Example: Feature Layer Query Results to a Spatial DataFrame](#Example:-Feature-Layer-Query-Results-to-a-Spatial-DataFrame)
* [Accessing Feature Class Data](#Accessing-Feature-Class-Data)
 * [Example: Reading a Shapefile](#Example:-Reading-a-Shapefile)
* [Saving Spatial DataFrames](#Saving-Spatial-DataFrames)
 * [Export Options](#Export-Options)
 * [Exporting to a Feature Class](#Export-to-Feature-Class)
   * [Example: Export a whole dataset to a Geodatabase](#Example:-Export-a-whole-dataset-to-a-geodatabase-feature-class:)
   * [Example: Export a subset to a Shapefile](#Example:-Export-dataset-with-a-subset-of-columns-and-top-5-records-to-a-shapefile:)
* [Visualizing Spatial Data](#Visualizing-Spatial-Data)
 * [Color Maps and Colors](#Color-Maps-and-Colors)
 * [Renderers](#Renderers)
 * [Symbology for Simple Renderers](#Symbology-for-Simple-Renderers)
* [Advanced Topics](#Advanced-Topics)
 * [Spatial Indexing](#Spatial-Indexing)
 * [Spatial Joins](#Spatial-Joins)
   * [Example: Merging State Statistics with Cities](#Example:-Merging-State-Statistics-Information-with-Cities)
 * [Spatial DataFrame Serialization](#Spatial-DataFrame-Serialization)
   * [Pickling the Spatial DataFrame](#Pickling-the-Spatial-DataFrame)
   * [Unpickling the Spatial DataFrame](#Unpickling-the-Spatial-DataFrame)

In [1]:
from arcgis.features import SpatialDataFrame

## Accessing Published Data

Software users need to work with both published data on remote servers and local data, but the ability to manipulate these datasets without permanentently copying the data is lacking.  The `Spatial DataFrame` solves this problem because it is an in-memory object that can read, write and manipulate geospatial data. 

The SDF integrates with Esri's [`ArcPy site-package`](http://pro.arcgis.com/en/pro-app/arcpy/get-started/what-is-arcpy-.htm) as well as the open source [`pyshp`](https://github.com/GeospatialPython/pyshp/), [`shapely`](https://github.com/Toblerity/Shapely) and [`fiona`](https://github.com/Toblerity/Fiona) packages. This means the ArcGIS API for Python SDF can use either of these geometry engines to provide you options for easily working with geospatial data regardless of your platform.  The SDF transforms data into the formats you desire so you can use Python functionality to analyze and visualize geographic information.  

Data can be read and scripted to automate workflows and just as easily visualized on maps in [`Jupyter notebooks`](../using-the-jupyter-notebook-environment/). The SDF can export data as feature classes or publish them directly to servers for sharing according to your needs.

Let's explore some of the different options available with the versatile `SpatialDataFrame` object:

### Reading Service Data

[`Feature layers`](https://doc.arcgis.com/en/arcgis-online/share-maps/hosted-web-layers.htm) hosted on [**ArcGIS Online**](https://www.arcgis.com) provide a wealth of information accessible from any device. However, any individual user will want to format the information to suit his or her needs.  The SDF has a [`from_layer`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#arcgis.features.SpatialDataFrame.from_layer) method that allows for consuming a `feature layer`, reporting data from it, and manipulating the data to a form that's comfortable and makes sense for the intended purpose.

**Example: Retrieving an ArcGIS Online [`item`](https://developers.arcgis.com/rest/users-groups-and-items/publish-item.htm) and inspecting its first 5 records**

In [2]:
from arcgis import GIS
item = GIS().content.get("85d0ca4ea1ca4b9abf0c51b9bd34de2e")
layer = item.layers[0]
sdf = SpatialDataFrame.from_layer(layer)
sdf.head()

,AGE_10_14,AGE_15_19,AGE_20_24,AGE_25_34,AGE_35_44,AGE_45_54,AGE_55_64,AGE_5_9,AGE_65_74,AGE_75_84,...,PLACEFIPS,POP2010,POPULATION,POP_CLASS,RENTER_OCC,ST,STFIPS,VACANT,WHITE,SHAPE
0,2144,2314,2002,3531,3887,5643,6353,2067,5799,2850,...,0408220,39540,40346,6,6563,AZ,04,6703,32367,"{'x': -12751215.004681978, 'y': 4180278.406256..."
1,876,867,574,1247,1560,2122,2342,733,2157,975,...,0424895,14364,14847,6,1397,AZ,04,1389,12730,"{'x': -12755627.731115643, 'y': 4164465.572856..."
2,1000,1003,833,2311,2063,2374,3631,1068,6165,3776,...,0425030,26265,26977,6,1963,AZ,04,9636,22995,"{'x': -12734674.294574209, 'y': 3850472.723091..."
3,2730,2850,2194,4674,5240,7438,8440,2499,8145,4608,...,0439370,52527,55041,7,6765,AZ,04,9159,47335,"{'x': -12725332.21151233, 'y': 4096532.0908223..."
4,2732,2965,2024,3182,3512,3109,1632,2497,916,467,...,0463470,25505,29767,6,1681,AZ,04,572,16120,"{'x': -12770984.257542243, 'y': 3826624.133935..."


### Reading Feature Layer Data

The SDF can consume a `Feature Layer` service accessible on the ArcGIS Online platform. You can log into the platform anonymously and retrieve data that is publicly available.


#### Example: Examining Feature Layer content

In [3]:
from arcgis.gis import GIS
gis = GIS()
item = gis.content.get("85d0ca4ea1ca4b9abf0c51b9bd34de2e")
fl = item.layers[0]
item

<Item title:"USA Major Cities" type:Feature Layer Collection owner:esri_dm>

Use the `from_layer` method on the SDF to instantiate a data frame from an item's `layer` and inspect the first 5 records.

In [4]:
sdf = SpatialDataFrame.from_layer(fl)
sdf.head()

,AGE_10_14,AGE_15_19,AGE_20_24,AGE_25_34,AGE_35_44,AGE_45_54,AGE_55_64,AGE_5_9,AGE_65_74,AGE_75_84,...,PLACEFIPS,POP2010,POPULATION,POP_CLASS,RENTER_OCC,ST,STFIPS,VACANT,WHITE,SHAPE
0,2144,2314,2002,3531,3887,5643,6353,2067,5799,2850,...,0408220,39540,40346,6,6563,AZ,04,6703,32367,"{'x': -12751215.004681978, 'y': 4180278.406256..."
1,876,867,574,1247,1560,2122,2342,733,2157,975,...,0424895,14364,14847,6,1397,AZ,04,1389,12730,"{'x': -12755627.731115643, 'y': 4164465.572856..."
2,1000,1003,833,2311,2063,2374,3631,1068,6165,3776,...,0425030,26265,26977,6,1963,AZ,04,9636,22995,"{'x': -12734674.294574209, 'y': 3850472.723091..."
3,2730,2850,2194,4674,5240,7438,8440,2499,8145,4608,...,0439370,52527,55041,7,6765,AZ,04,9159,47335,"{'x': -12725332.21151233, 'y': 4096532.0908223..."
4,2732,2965,2024,3182,3512,3109,1632,2497,916,467,...,0463470,25505,29767,6,1681,AZ,04,572,16120,"{'x': -12770984.257542243, 'y': 3826624.133935..."


You can refine sql queries by leveraging the ArcGIS API for Python's [`Feature Layer`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#featurelayer) object itself to return a subset of records. Instantiate a `Pandas data frame` directly from the [`feature layer.query`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#arcgis.features.FeatureLayer.query) and use the data frame's [`head`](http://pandas.pydata.org/pandas-docs/stable/generated/pandas.core.groupby.GroupBy.head.html#pandas.core.groupby.GroupBy.head) method to return the first 5 records:

#### Example: Feature Layer Query Results to a Spatial DataFrame

In [5]:
df = fl.query(where="AGE_45_54 < 1500").df
df[['NAME', 'AGE_45_54', 'POP2010']].head()

,NAME,AGE_45_54,POP2010
0,Somerton,1411,14287
1,Anderson,1333,9932
2,Camp Pendleton South,127,10616
3,Citrus,1443,10866
4,Commerce,1478,12823


## Accessing Feature Class Data

The SDF can also access local geospatial data. Depending upon what Python modules you have installed, you'll have access to a wide range of functionality:  

* If the **`ArcPy`** module is installed, meaning you have installed either [`ArcGIS Desktop`](http://desktop.arcgis.com/en/) or [`ArcGIS Pro`](http://pro.arcgis.com/en/pro-app/) (or both) and have installed the ArcGIS API for Python in that same environment, the Spatial DataFrame has methods to read a subset of the ArcGIS Desktop [supported geographic formats](http://desktop.arcgis.com/en/arcmap/10.3/manage-data/datatypes/about-geographic-data-formats.htm#ESRI_SECTION1_4835793C55C0439593A46FD5BC9E64B9), most notably:
 * [`feature classes`](http://desktop.arcgis.com/en/arcmap/latest/manage-data/feature-classes/a-quick-tour-of-feature-classes.htm)
 * [`shapefiles`](http://desktop.arcgis.com/en/arcmap/latest/manage-data/shapefiles/what-is-a-shapefile.htm),  
 * [`ArcGIS Server Web Services`](https://enterprise.arcgis.com/en/server/latest/publish-services/windows/what-types-of-services-can-you-publish.htm) and [`ArcGIS Online Hosted Feature Layers`](https://doc.arcgis.com/en/arcgis-online/share-maps/publish-features.htm) 
 * [`OGC Services`](http://www.opengeospatial.org/standards)  
* If the **ArcPy** module is not installed, the SDF [`from_featureclass`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#arcgis.features.SpatialDataFrame.from_featureclass) method only supports consuming an Esri [`shapefile`](http://desktop.arcgis.com/en/arcmap/latest/manage-data/shapefiles/what-is-a-shapefile.htm)
> Please note that you must install the `pyshp` module to read shapefiles with Python interpreters that don't have access to `ArcPy`.
    
### Example: Reading a Shapefile
> You must authenticate to `ArcGIS Online` or `ArcGIS Enterprise` to read a shapefile with a Python interpreter that does not have access to `ArcPy` (either an ArcGIS Desktop or ArcGIS Pro installation).

>  `g2 = GIS("https://www.arcgis.com", "username", "password")`

In [9]:
sdf = SpatialDataFrame.from_featureclass("./data/Census2010_Tracts.shp")
sdf.tail()

,index,AGE_18_21,AGE_22_29,AGE_30_39,AGE_40_49,AGE_50_64,AGE_5_17,AGE_65_UP,AGE_UNDER5,AMERI_ES,...,POP2000,POP2010,RENTER_OCC,SHAPE,SQMI,STATE_FIPS,STCOFIPS,TRACT,VACANT,WHITE
26916,26916,286,592,684,642,651,582,966,294,160,...,4697,4660,1458,"{'rings': (((-88.0462200780785, 44.51748096527...",0.94,55,55009,000401,70,4213
26917,26917,295,607,701,653,756,705,692,280,144,...,4689,4633,1182,"{'rings': (((-88.063086961631, 44.524976930835...",1.08,55,55009,000302,60,4316
26918,26918,262,566,730,574,437,872,366,364,226,...,4171,4082,718,"{'rings': (((-87.98274797423953, 44.5190819787...",1.26,55,55009,000900,85,2830
26919,26919,195,398,420,374,388,460,634,225,123,...,3094,3060,712,"{'rings': (((-88.06053994789228, 44.5287709600...",0.73,55,55009,000303,28,2726
26920,26920,183,449,807,746,712,887,596,271,12,...,4651,4677,456,"{'rings': (((-73.26635396725263, 44.5394849978...",1.49,50,50007,000100,39,4361


## Saving Spatial DataFrames

The SDF can export data to various data formats for use in other applications.


### Export Options

- [Feature Layers](https://doc.arcgis.com/en/arcgis-online/share-maps/hosted-web-layers.htm)
- [Feature Collections](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#featurelayercollection)
- [Feature Set](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#featureset)
- [GeoJSON](http://geojson.org/)
- [Feature Class](http://desktop.arcgis.com/en/arcmap/latest/manage-data/feature-classes/a-quick-tour-of-feature-classes.htm)
- [Pickle](https://pythontips.com/2013/08/02/what-is-pickle-in-python/)
- [HDF](https://support.hdfgroup.org/HDF5/Tutor/HDF5Intro.pdf)

### Export to Feature Class

The SDF allows for the export of whole datasets or partial datasets.  

#### Example: Export a whole dataset to a geodatabase feature class:

In [ ]:
sdf.to_featureclass((out_location=r"c:\temp\scratch.gdb", out_name="myexport")

> The ArcGIS API for Python on all `macOS` and `Linux` machines, as well as those `Windows` machines not using Python interpreters that access ArcGIS Desktop or ArcGIS Pro installations will only be able to write out to shapefile format with the `to_featureclass` method. Writing to file geodatabases requires the `ArcPY` site-package.

#### Example: Export dataset with a subset of columns and top 5 records to a shapefile:

In [10]:
columns = [ 'STATE_FIPS', 'CNTY_FIPS', 'TRACT', 'POP2010', 'SQMI', 'SHAPE']
sdf[columns].head().to_featureclass(out_location=r"c:\temp",
                                    out_name="export.shp")

'c:\\temp/export.shp'

In [11]:
df = fl.query(where="AGE_45_54 < 1500").df
df[['NAME', 'AGE_45_54', 'POP2010']].head()

,NAME,AGE_45_54,POP2010
0,Somerton,1411,14287
1,Anderson,1333,9932
2,Camp Pendleton South,127,10616
3,Citrus,1443,10866
4,Commerce,1478,12823


## Visualizing Spatial Data
Some unique characteristics of working with the visualization capabalities on the SDF:
- Uses Pythonic syntax
- Uses symbology familiar to users of [`matplotlib`](https://matplotlib.org/)
- Works on features and attributes simultaneously, eliminating to a great extent the need to iterate over all features (rows)
- Handles reading and writing to multiple formats aiding data conversion

Dataset symbology can be customized using the `plot` method on a `Spatial DataFrame` or a `Pandas data frame`, as can be created from the [`FeaturSet`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#arcgis.features.FeatureSet) object that results from a [`FeatureLayer.query()`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#arcgis.features.FeatureLayer.query) operation.

In [ ]:
m = GIS().map('United States')
m

![map of US diamonds](http://esri.github.io/arcgis-python-api/notebooks/nbimages/13_3_map_dfplot1.png)

In [ ]:
m.center = [39, -98]
m.zoom = 4

The code below plots a set of points on the map above using a common language amongst many different Python packages for defining symbology. It is built off of the matplotlib libraries for simple, straightforward plotting. See the [`matplotlib.pyplot`](https://matplotlib.org/api/pyplot_summary.html) documentation as a starting point for further details.

In [ ]:
df.plot(kind='map', 
        map_widget=m,
        colors='Reds_r',
        marker_size=10,
        cstep=60,
        outline_color='Blues',
        symbol_style='d')

### Color Maps and Colors

Color specifications can be input to the `plot` meathod as a string representing [named colors](https://matplotlib.org/examples/color/named_colors.html), an array of [RGB](http://www.tomjewett.com/colors/rgb.html) values, or a named [color ramp](https://matplotlib.org/examples/color/colormaps_reference.html). 

#### Color Array

RGB and Alpha values can be used to create the symbols called in the `plot` method.  RGB stands for red, green, and blue respectively. Each RGB value is a value between 0-255, and the alpha value is a number between 0-255.

**Example to produce :**

    color = [255,0,100,1]

The above example produces a purplish color. Many websites provide details about using colors. For example, see [here](https://web.njit.edu/~kevin/rgt.txt.html#top) for color information categorized by shades.

#### Color Maps

A color map is a collection of string values that can be given to generate a series of related colors from a defined set.

Color maps can be viewed here: https://matplotlib.org/examples/color/colormaps_reference.html

#### Color Map Helpers

To better understand the syntax for each input type, the ArcGIS API for Python provides some helper functions.

In [ ]:
from arcgis.mapping import display_colormaps

In [ ]:
display_colormaps()

![named color ramps](http://esri.github.io/arcgis-python-api/notebooks/nbimages/13_display_colormaps.png)

The **display_colormaps** provides a quick, easy way to visualize the pre-defined set of colormaps you can use.  You can enter a list of color ramp names as input to the `display_colormaps` function to filter the output:

In [ ]:
display_colormaps(['Greens_r', 'PRGn', 'Dark2', 'Set1'])

![selected color ramps](http://esri.github.io/arcgis-python-api/notebooks/nbimages/13_selected_colormaps.png)

### Renderers
[`Renderers`](https://developers.arcgis.com/documentation/common-data-types/renderer-objects.htm) define how to visually represent a `feature layer` by defining [`symbols`](https://developers.arcgis.com/documentation/common-data-types/symbol-objects.htm) to represent individual features. The SDF provides you with functionality to control the way features appear by choosing the `symbol` the renderer uses.

Previous versions of the ArcGIS API for Python provided a method to specify a renderer manually, but you had to know details about the renderer before you drew your data. The [`map.add_layer()`]() method did not provide access to all options avialble for rendering datasets. The new visualization capabilities provided by the SDF allow you to draw spatial data quickly and easily with access to more rendering options.

#### Supported Renderers
The renderering options below are documented in further detail in the [`Spatial DatFrame Reference`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#spatialdataframe):

+ Simple - renders using one symbol only
+ Unique - renders on one or more string attributes
+ Class Breaks - renders on numeric data
+ Heatmap- renders point data into raster visualization

#### Renderer Syntax

+ 's' - is a simple renderer that uses one symbol only.
+ 'u' - unique renderer symbolizes features based on one or more matching string attributes.
+ 'c' - A class breaks renderer symbolizes based on the value of some numeric attribute.
+ 'h' - heatmap renders point data into a raster visualization that emphasizes areas of higher density or weighted values.

#### Using Renderers

Renderers are generated when on Spatial DataFrames when the [`plot`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#arcgis.features.SpatialDataFrame.plot) method is called.

### Symbology for Simple Renderers

The ArcGIS API for Python provides you the ability to set symbol types so you control data appearance. The  [`show_styles`]() function in the `arcgis.mapping` module assists developers with the syntax to define symbols.

#### Getting the Symbol Style


In [12]:
from arcgis.mapping import show_styles

In [13]:
show_styles(sdf.geometry_type)

,MARKER,ESRI_STYLE
0,\,Backward Diagonal
1,/,Forward Diagonal
2,|,Vertical Bar
3,-,Horizontal Bar
4,x,Diagonal Cross
5,+,Cross
6,s,Solid Fill (default)


You can define symbols within the plot method of the Spatial DataFrame..

In [ ]:
m = GIS().map('United States', 4)
m

![map of US squares](http://esri.github.io/arcgis-python-api/notebooks/nbimages/13_4_map_dfplot2.png

In [ ]:
m.zoom = 4
m.center = [39, -98]
df.plot(map_widget=m,
        kind='map',
        symbol_style='s',
        marker_size=5,
        line_width=.5,
        pallette='Set1',
        outline_color='Greens_r')

# Advanced Topics

The information beyond this section provides a brief introduction to advanced topics with the `Spatial DataFrame` structure.  

## Spatial Indexing

The Spatial DataFrame uses [QuadTree indexing](https://en.wikipedia.org/wiki/Quadtree) on the geometries to aid in spatial querying. In the [**Examining Feature Layer content**](#Example:-Examining-Feature-Layer-content) section of this notebook, the USA Major Cities feature layer was queried and the `df` method was called on the results to create a data frame. The [`sindex`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.html?highlight=style#arcgis.features.SpatialDataFrame.sindex) method on the `df` creates a quad tree index.

The `intersect` method of the resulting index takes a bounding box as input (4 coordinates representing the minimum and maximum x,y coordinate pairs) and returns an object that can be converted into a list of features that  intersect that box.

In [14]:
si = df.sindex
index = si.intersect([-13143219.122301877-100000, 4011134.034258818-100000, 
                      -13143219.102301877+100000, 4011134.0542588173+100000])

In [15]:
df.iloc[list(index)]

,AGE_10_14,AGE_15_19,AGE_20_24,AGE_25_34,AGE_35_44,AGE_45_54,AGE_55_64,AGE_5_9,AGE_65_74,AGE_75_84,...,PLACEFIPS,POP2010,POPULATION,POP_CLASS,RENTER_OCC,ST,STFIPS,VACANT,WHITE,SHAPE
2,593,511,2323,2767,746,127,34,1229,4,2,...,0610561,10616,11869,6,2558,CA,06,296,7530,"{'x': -13066582.116550362, 'y': 3925650.676616..."
3,888,988,900,1729,1479,1443,959,766,514,280,...,0613560,10866,11195,6,761,CA,06,86,5898,"{'x': -13123874.446103057, 'y': 4044249.710416..."
4,1086,1228,1013,1822,1759,1478,1112,925,687,477,...,0614974,12823,13009,6,1763,CA,06,88,6930,"{'x': -13151212.145276317, 'y': 4027601.332347..."
10,1076,1118,952,1707,1651,1397,931,955,524,258,...,0634302,11570,12025,6,808,CA,06,105,5275,"{'x': -13080593.062843386, 'y': 4012557.378155..."
14,14,24,44,113,153,523,2425,11,3883,5122,...,0639259,16192,17499,6,2572,CA,06,1714,14133,"{'x': -13105436.040130444, 'y': 3977081.952377..."
22,1041,1219,1041,1402,1317,1271,727,996,396,145,...,0650132,10644,11092,6,963,CA,06,212,4459,"{'x': -13063142.772085657, 'y': 4049606.978283..."


## Spatial Joins

A Spatial join is a GIS operation that affixes data from one feature layer’s attribute table to another based on a spatial relationship.

The spatial join involves matching rows from the Join Features (data frame1) to the Target Features (data frame2) based on their relative spatial locations.  

In [16]:
from arcgis.features._data.geodataset.tools import spatial_join

#### Example: Merging State Statistics Information with Cities

The goal is get Hawaii's city locations joined with Hawaii's state census data.
> If you do not access to the `ArcPy` site-package from the Python interpreter used to execute the following cells, you must authenticate to an ArcGIS Online Organization or ArcGIS Enterprise portal.

> g3 = GIS("https://www.arcgis.com", "username", "password")

In [ ]:
sdf_target = SpatialDataFrame.from_featureclass(r"C:\temp\scratch.gdb\cities1")
sdf_join = SpatialDataFrame.from_featureclass(r"C:\temp\states.gdb\states")

In [ ]:
sdf_target.geometry_type, sdf_join.geometry_type
q = sdf_target['ST'] == 'HI'
left = sdf_target[q].copy()
left.head()

,OBJECTID,NAME,CLASS,ST,STFIPS,PLACEFIP,CAPITAL,AREALAND,AREAWATER,POP_CLASS,...,MARHH_NO_C,MHH_CHILD,FHH_CHILD,FAMILIES,AVE_FAM_SZ,HSE_UNITS,VACANT,OWNER_OCC,RENTER_OCC,SHAPE
9,10,Hilo,Census Designated Place,HI,15,14650,,54.289,4.147,6,...,4227,388,1228,10105,3.19,16026,1449,8873,5704,"{'x': -155.08314566813806, 'y': 19.70248163811..."
10,11,Kahului,Census Designated Place,HI,15,22700,,15.162,1.175,6,...,1682,185,495,4424,3.76,6079,199,3190,2690,"{'x': -156.46462715222128, 'y': 20.87854563897..."
11,12,Kihei,Census Designated Place,HI,15,36500,,10.159,1.731,6,...,1457,201,413,3811,3.31,9170,3000,3007,3163,"{'x': -156.45440916395174, 'y': 20.75592714831..."
12,13,Wailuku,Census Designated Place,HI,15,77450,,5.066,0.374,6,...,1246,104,268,3016,3.28,4780,245,2675,1860,"{'x': -156.4993546618881, 'y': 20.888718129638..."
13,14,Ewa Beach,Census Designated Place,HI,15,07450,,1.417,0.449,6,...,1216,83,227,2941,4.47,3515,210,2278,1027,"{'x': -158.00898167116716, 'y': 21.31570015363..."


In [ ]:
q = sdf_join.STATE_ABBR == 'HI'
right = sdf_join[q].copy()
right.head()

,OBJECTID,STATE_NAME,STATE_FIPS,SUB_REGION,STATE_ABBR,POP2000,POP2010,POP00_SQMI,POP10_SQMI,WHITE,...,HSE_UNITS,VACANT,OWNER_OCC,RENTER_OCC,NO_FARMS07,AVG_SIZE07,CROP_ACR07,AVG_SALE07,SQMI,SHAPE
0,1,Hawaii,15,Pacific,HI,1211537,1309580,110.8,119.8,294102,...,460542,57302,227888,175352,7521.0,149.0,177626.0,68.29,10931,"{'rings': [[[-160.07380355019365, 22.004177304..."


In [ ]:
sdf = spatial_join(df1=left, df2=right)
sdf.head()

,TARGET_OID,JOIN_OID,OBJECTID_left,NAME,CLASS,ST,STFIPS,PLACEFIP,CAPITAL,AREALAND,...,AVE_FAM_SZ_right,HSE_UNITS_right,VACANT_right,OWNER_OCC_right,RENTER_OCC_right,NO_FARMS07,AVG_SIZE07,CROP_ACR07,AVG_SALE07,SQMI
0,9,0,10,Hilo,Census Designated Place,HI,15,14650,,54.289,...,3.42,460542.0,57302.0,227888.0,175352.0,7521.0,149.0,177626.0,68.29,10931.0
1,10,0,11,Kahului,Census Designated Place,HI,15,22700,,15.162,...,3.42,460542.0,57302.0,227888.0,175352.0,7521.0,149.0,177626.0,68.29,10931.0
2,11,0,12,Kihei,Census Designated Place,HI,15,36500,,10.159,...,3.42,460542.0,57302.0,227888.0,175352.0,7521.0,149.0,177626.0,68.29,10931.0
3,12,0,13,Wailuku,Census Designated Place,HI,15,77450,,5.066,...,3.42,460542.0,57302.0,227888.0,175352.0,7521.0,149.0,177626.0,68.29,10931.0
4,13,0,14,Ewa Beach,Census Designated Place,HI,15,07450,,1.417,...,3.42,460542.0,57302.0,227888.0,175352.0,7521.0,149.0,177626.0,68.29,10931.0


## Spatial DataFrame Serialization

From the python.org help:

    The pickle module implements a fundamental, but powerful algorithm for serializing and de-serializing a Python object structure. 'Pickling' is the process whereby a Python object hierarchy is converted into a byte stream, and 'unpickling' is the inverse operation, whereby a byte stream is converted back into an object hierarchy. Pickling (and unpickling) is alternatively known as 'serialization', 'marshalling', or 'flattening', however, to avoid confusion, the terms used here are 'pickling' and 'unpickling'.


The Spatial DataFrame supports this operation, and it's a powerful way to share and store the current state of the Spatial DataFrame without having to export it out to another format.

### Pickling the Spatial DataFrame

In [ ]:
import pickle
out_pickle = r"c:\temp\mysdf.pkl"
sdf.to_pickle(out_pickle)

Now that our file has been created, the unpickling process can occur.

### Unpickling the Spatial DataFrame

In [ ]:
unpickled_sdf = pickle.loads(open(out_pickle, 'rb').read()).head()
unpickled_sdf.head()